<a href="https://colab.research.google.com/github/DharaCS23181/Skincare_Product_Prediction/blob/main/XGBoost80%E2%80%9320.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install xgboost

In [6]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from xgboost import XGBClassifier

In [7]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv("/content/drive/MyDrive/ML Dataset/skin_recommendation_dataset.csv")
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,skintype,skin_condition,product_type,product_name,brand,notable_effects,picture_src
0,Oily,"('wrinkles', 'pigmentation', 'acne', 'enlarged...",Face Wash,ACWELL Bubble Free PH Balancing Cleanser,ACWELL,"('acne-free', 'pore-care', 'brightening', 'ant...",https://www.beautyhaul.com/assets/uploads/prod...
1,"Normal, Dry, Combination","('redness', 'skin imbalance')",Face Wash,ACWELL pH Balancing Soothing Cleansing Foam,ACWELL,"('soothing', 'balancing')",https://images.soco.id/8f08ced0-344d-41f4-a15e...
2,"Normal, Dry, Oily, Combination, Sensitive","('redness', 'skin imbalance')",Toner,Acwell Licorice pH Balancing Cleansing Toner,ACWELL,"('soothing', 'balancing')","https://www.soco.id/cdn-cgi/image/w=73,format=..."
3,Oily,"('wrinkles', 'pigmentation', 'acne', 'enlarged...",Toner,ACWELL Aquaseal Soothing Tonic,ACWELL,"('acne-free', 'pore-care', 'brightening', 'ant...",https://www.beautyhaul.com/assets/uploads/prod...
4,"Normal, Dry","('pigmentation', 'redness')",Toner,Licorice pH Balancing Essence Mist,ACWELL,"('brightening', 'soothing')","https://www.sociolla.com/cdn-cgi/image/w=425,f..."


In [8]:
data = df[['skintype','skin_condition','notable_effects','product_type']]
data.head()

,skintype,skin_condition,notable_effects,product_type
0,Oily,"('wrinkles', 'pigmentation', 'acne', 'enlarged...","('acne-free', 'pore-care', 'brightening', 'ant...",Face Wash
1,"Normal, Dry, Combination","('redness', 'skin imbalance')","('soothing', 'balancing')",Face Wash
2,"Normal, Dry, Oily, Combination, Sensitive","('redness', 'skin imbalance')","('soothing', 'balancing')",Toner
3,Oily,"('wrinkles', 'pigmentation', 'acne', 'enlarged...","('acne-free', 'pore-care', 'brightening', 'ant...",Toner
4,"Normal, Dry","('pigmentation', 'redness')","('brightening', 'soothing')",Toner


In [9]:
for col in ['skintype', 'skin_condition', 'notable_effects']:
    data[col] = data[col].apply(lambda x: x.split(', '))

/tmp/ipykernel_4386/661583364.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[col] = data[col].apply(lambda x: x.split(', '))


In [10]:
mlb_skin = MultiLabelBinarizer()
mlb_condition = MultiLabelBinarizer()
mlb_effects = MultiLabelBinarizer()

skin_features = pd.DataFrame(mlb_skin.fit_transform(data['skintype']))
condition_features = pd.DataFrame(mlb_condition.fit_transform(data['skin_condition']))
effects_features = pd.DataFrame(mlb_effects.fit_transform(data['notable_effects']))

In [11]:
le = LabelEncoder()
y = le.fit_transform(data['product_type'])

In [12]:
X = pd.concat([skin_features, condition_features, effects_features], axis=1)

In [13]:
X_train_80, X_test_20, y_train_80, y_test_20 = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [16]:
xgb_80 = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)

xgb_80.fit(X_train_80.to_numpy(), y_train_80)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.9, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, ...)

In [20]:
y_pred_80 = xgb_80.predict(X_test_20.to_numpy())

print("XGBoost (80-20) Accuracy:", accuracy_score(y_test_20, y_pred_80))
print(classification_report(y_test_20, y_pred_80))

XGBoost (80-20) Accuracy: 0.5510204081632653
              precision    recall  f1-score   support

           0       0.57      0.53      0.55        40
           1       0.47      0.52      0.50        50
           2       0.59      0.69      0.64        62
           3       0.71      0.60      0.65        42
           4       0.44      0.39      0.42        51

    accuracy                           0.55       245
   macro avg       0.56      0.55      0.55       245
weighted avg       0.55      0.55      0.55       245



In [21]:
new_user = X.iloc[0:1]

prediction = xgb_80.predict(new_user.to_numpy())
recommended_product = le.inverse_transform(prediction)

print("Recommended Product:", recommended_product)

Recommended Product: ['Serum']
